# Digital Clock - Multiple Time Zones
This application displays the current time in different time zones with a real-time update feature.

In [ ]:
import datetime
import pytz
from IPython.display import clear_output, display, HTML
import time
from ipywidgets import Output, VBox, HBox, Label, Button
import ipywidgets as widgets

## Define Time Zones

In [ ]:
# Define the time zones you want to display
time_zones = {
    'UTC': 'UTC',
    'EST (New York)': 'US/Eastern',
    'CST (Chicago)': 'US/Central',
    'MST (Denver)': 'US/Mountain',
    'PST (Los Angeles)': 'US/Pacific',
    'GMT (London)': 'Europe/London',
    'CET (Paris)': 'Europe/Paris',
    'IST (India)': 'Asia/Kolkata',
    'JST (Tokyo)': 'Asia/Tokyo',
    'AEST (Sydney)': 'Australia/Sydney'
}

print("Available Time Zones:")
for label, tz in time_zones.items():
    print(f"  - {label}: {tz}")

## Create Digital Clock Display

In [ ]:
def get_current_time_in_timezone(timezone_str):
    """Get current time in a specific timezone."""
    tz = pytz.timezone(timezone_str)
    current_time = datetime.datetime.now(tz)
    return current_time.strftime('%H:%M:%S')

def create_clock_display():
    """Create HTML display for the digital clock."""
    html_content = """
    <style>
        .clock-container {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
            gap: 20px;
            padding: 20px;
            background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);
            border-radius: 10px;
        }
        
        .clock-card {
            background: #fff;
            border-radius: 8px;
            padding: 20px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
            text-align: center;
            transition: transform 0.3s ease;
        }
        
        .clock-card:hover {
            transform: translateY(-5px);
            box-shadow: 0 8px 12px rgba(0, 0, 0, 0.2);
        }
        
        .timezone-label {
            font-size: 14px;
            color: #666;
            font-weight: 600;
            margin-bottom: 10px;
            text-transform: uppercase;
            letter-spacing: 1px;
        }
        
        .time-display {
            font-size: 32px;
            font-weight: bold;
            color: #2a5298;
            font-family: 'Courier New', monospace;
            letter-spacing: 2px;
        }
        
        .date-display {
            font-size: 12px;
            color: #999;
            margin-top: 10px;
            font-family: 'Courier New', monospace;
        }
    </style>
    
    <div class="clock-container">
    """
    
    for label, timezone in time_zones.items():
        current_time = get_current_time_in_timezone(timezone)
        tz = pytz.timezone(timezone)
        current_date = datetime.datetime.now(tz).strftime('%a, %b %d, %Y')
        
        html_content += f"""
        <div class="clock-card">
            <div class="timezone-label">{label}</div>
            <div class="time-display">{current_time}</div>
            <div class="date-display">{current_date}</div>
        </div>
        """
    
    html_content += "</div>"
    return html_content

# Display the clock
display(HTML(create_clock_display()))

## Live Updating Clock (Interactive)

In [ ]:
# Create output widget for live updates
output = Output()
stop_button = Button(description='Stop Clock', button_style='danger')
start_button = Button(description='Start Clock', button_style='success')

clock_running = {'value': False}

def update_clock():
    """Update the clock display continuously."""
    with output:
        while clock_running['value']:
            clear_output(wait=True)
            display(HTML(create_clock_display()))
            time.sleep(1)

def start_clock(button):
    clock_running['value'] = True
    update_clock()

def stop_clock(button):
    clock_running['value'] = False
    with output:
        clear_output()
        print("Clock stopped.")

start_button.on_click(start_clock)
stop_button.on_click(stop_clock)

button_box = HBox([start_button, stop_button])
display(button_box)
display(output)

## Get Specific Timezone Time

In [ ]:
def get_time_by_timezone(timezone_name):
    """Get the current time for a specific timezone.
    
    Args:
        timezone_name (str): Name of the timezone (e.g., 'US/Eastern', 'Asia/Tokyo')
    
    Returns:
        dict: Contains time, date, and timezone info
    """
    try:
        tz = pytz.timezone(timezone_name)
        current_time = datetime.datetime.now(tz)
        
        return {
            'timezone': timezone_name,
            'time': current_time.strftime('%H:%M:%S'),
            'date': current_time.strftime('%A, %B %d, %Y'),
            'offset': current_time.strftime('%z'),
            'full_datetime': current_time.strftime('%Y-%m-%d %H:%M:%S %Z')
        }
    except Exception as e:
        return {'error': f'Invalid timezone: {timezone_name}'}

# Example usage
tokyo_time = get_time_by_timezone('Asia/Tokyo')
print("\nTokyo Time:")
for key, value in tokyo_time.items():
    print(f"  {key}: {value}")

sydney_time = get_time_by_timezone('Australia/Sydney')
print("\nSydney Time:")
for key, value in sydney_time.items():
    print(f"  {key}: {value}")

## Timezone Conversion Helper

In [ ]:
def convert_time(source_timezone, target_timezone, time_string=None):
    """Convert time from one timezone to another.
    
    Args:
        source_timezone (str): Source timezone name
        target_timezone (str): Target timezone name
        time_string (str): Time in format 'HH:MM:SS' (if None, uses current time)
    
    Returns:
        str: Time in target timezone
    """
    try:
        source_tz = pytz.timezone(source_timezone)
        target_tz = pytz.timezone(target_timezone)
        
        if time_string is None:
            # Use current time
            current_time = datetime.datetime.now(source_tz)
        else:
            # Parse provided time
            time_obj = datetime.datetime.strptime(time_string, '%H:%M:%S')
            current_time = source_tz.localize(time_obj)
        
        converted_time = current_time.astimezone(target_tz)
        return converted_time.strftime('%H:%M:%S')
    except Exception as e:
        return f'Error: {str(e)}'

# Example: Convert current time from New York to Tokyo
ny_to_tokyo = convert_time('US/Eastern', 'Asia/Tokyo')
print(f"Current time in New York converted to Tokyo: {ny_to_tokyo}")

# Example: Convert a specific time
specific_time = convert_time('Europe/London', 'Australia/Sydney', '12:00:00')
print(f"12:00:00 in London converted to Sydney: {specific_time}")

## List All Available Timezones

In [ ]:
# Get all available timezones
all_timezones = pytz.all_timezones
print(f"Total available timezones: {len(all_timezones)}\n")
print("First 20 timezones:")
for i, tz in enumerate(all_timezones[:20]):
    print(f"  {i+1}. {tz}")